# Seleção de modelo

- separa holdout estratificado (20%)
- nested CV no treino (BayesSearchCV no inner, avaliação no outer)
- escolhe threshold por OOF no treino
- avalia no holdout e calcula IC por bootstrap
- salva CSV/JSON com resultados


In [ ]:
%pip install -q scikit-optimize xgboost catboost openpyxl


In [ ]:
import os
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, RepeatedStratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.base import clone, ClassifierMixin
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, balanced_accuracy_score,
    matthews_corrcoef, brier_score_loss,
)
from sklearn.utils import check_random_state

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*CatBoost.*")


In [ ]:
BASES_DIR = "bases"
BASE_FILE = "baseModelo3-FatoresAssociadosAoObito.xlsx"
TARGET_COL = "OBITO"

TEST_SIZE = 0.20
RANDOM_STATE = 42

OUTER_SPLITS = 10
OUTER_REPEATS = 1
INNER_SPLITS = 10

N_ITER_BAYES = 90
N_JOBS_BAYES = 1
VERBOSE_BAYES = 0

OPTIMIZE_METRIC = "average_precision"
THRESH_METRIC = "precision"   # "precision", "f1" ou "youden"
MIN_RECALL = 0.7

N_BOOTSTRAPS = 1000
CI_LEVEL = 0.95

OUTPUT_DIR = "resultados_selecao_modelo_base3"
SAVE_JSON = True
SAVE_CSV = True


In [ ]:
# O sklearn tinha gerado um erro com o catboost...
class SafeCatBoostClassifier(ClassifierMixin, CatBoostClassifier):
    _estimator_type = "classifier"

    # sklearn >= 1.6 usa tags; em versões antigas isso não é necessário
    def __sklearn_tags__(self):
        try:
            from sklearn.utils import Tags, TargetTags, ClassifierTags
            return Tags(
                estimator_type="classifier",
                target_tags=TargetTags(required=True),
                transformer_tags=None,
                classifier_tags=ClassifierTags(),
                regressor_tags=None,
            )
        except Exception:
            # sklearn < 1.6: devolve tags no formato antigo (dict)
            if hasattr(self, "_get_tags"):
                return self._get_tags()
            return {}


In [ ]:
def make_pipeline(estimator):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", clone(estimator)),
    ])


def compute_metrics(y_true, y_proba, y_pred):
    return {
        "roc_auc": float(roc_auc_score(y_true, y_proba)),
        "average_precision": float(average_precision_score(y_true, y_proba)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "brier": float(brier_score_loss(y_true, y_proba)),
    }


def select_threshold(y_true, proba, metric="f1", min_recall=None):
    thresholds = np.linspace(0.001, 0.999, 999)
    best_t, best_key = None, None  # (valor_metrica, recall, threshold)

    for t in thresholds:
        pred = (proba >= t).astype(int)

        rec = recall_score(y_true, pred, zero_division=0)
        if min_recall is not None and rec < min_recall:
            continue

        prec = precision_score(y_true, pred, zero_division=0)
        f1 = f1_score(y_true, pred, zero_division=0)

        if metric == "precision":
            val = prec
        elif metric == "f1":
            val = f1
        elif metric == "youden":
            tn = np.sum((y_true == 0) & (pred == 0))
            fp = np.sum((y_true == 0) & (pred == 1))
            spec = tn / (tn + fp + 1e-12)
            val = rec + spec - 1
        else:
            raise ValueError("metric deve ser 'precision', 'f1' ou 'youden'")

        key = (val, rec, float(t))
        if best_key is None or key > best_key:
            best_key = key
            best_t = float(t)

    if best_t is None:
        raise ValueError(f"Nenhum threshold atingiu min_recall={min_recall}.")
    return best_t, float(best_key[0])


def oof_predictions(pipeline, X_data, y_data, n_splits, random_state):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    out = np.full(len(y_data), -1.0, dtype=float)

    for i, (tr, te) in enumerate(cv.split(X_data, y_data)):
        est = clone(pipeline)
        rs = random_state + i

        try:
            try:
                est.set_params(model__random_seed=None)
            except Exception:
                pass
            est.set_params(model__random_state=rs)
        except Exception:
            try:
                est.set_params(model__random_seed=rs)
            except Exception:
                pass

        est.fit(X_data.iloc[tr], y_data[tr])
        out[te] = est.predict_proba(X_data.iloc[te])[:, 1]

    if (out < 0).any():
        raise RuntimeError("OOF incompleto.")
    return out


def bootstrap_ci(y_true, y_proba, y_pred, n_boot=1000, ci=0.95, random_state=42):
    rng = check_random_state(random_state)
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    y_pred = np.asarray(y_pred)

    idx0 = np.where(y_true == 0)[0]
    idx1 = np.where(y_true == 1)[0]

    keys = list(compute_metrics(y_true, y_proba, y_pred).keys())
    vals = {k: [] for k in keys}

    for _ in range(n_boot):
        s = np.concatenate([
            rng.choice(idx0, len(idx0), replace=True),
            rng.choice(idx1, len(idx1), replace=True),
        ])
        m = compute_metrics(y_true[s], y_proba[s], y_pred[s])
        for k in keys:
            vals[k].append(m[k])

    alpha = (1 - ci) / 2
    out = {}
    for k, arr in vals.items():
        arr = np.asarray(arr)
        out[k] = {
            "mean_boot": float(arr.mean()),
            "ci_low": float(np.quantile(arr, alpha)),
            "ci_high": float(np.quantile(arr, 1 - alpha)),
        }
    return out


In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

path1 = os.path.join(BASES_DIR, BASE_FILE)
base_path = path1 if os.path.exists(path1) else BASE_FILE

df = pd.read_excel(base_path)
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

n_pos_train = int(y_train.sum())
n_neg_train = int((y_train == 0).sum())
CLASS_RATIO_TRAIN = n_neg_train / max(n_pos_train, 1)

sw_train = np.where(y_train == 1, CLASS_RATIO_TRAIN, 1.0)
sw_train = sw_train / sw_train.mean()

split_info = {
    "base_file": BASE_FILE,
    "target_col": TARGET_COL,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "class_ratio_train": CLASS_RATIO_TRAIN,
    "train_shape": list(X_train.shape),
    "test_shape": list(X_test.shape),
    "train_index": X_train.index.tolist(),
    "test_index": X_test.index.tolist(),
    "created_at": datetime.now().isoformat(timespec="seconds"),
}
with open(os.path.join(OUTPUT_DIR, "holdout_split_indices.json"), "w", encoding="utf-8") as f:
    json.dump(split_info, f, ensure_ascii=False, indent=2)

print("Base:", BASE_FILE, "| X:", X.shape, "| pos:", int(y.sum()), "| neg:", int((y == 0).sum()))
print("Split salvo em:", os.path.join(OUTPUT_DIR, "holdout_split_indices.json"))


In [ ]:
model_specs = {
    "decision_tree": {
        "estimator": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        "search_space": {
            "model__max_depth": Integer(1, 30),
            "model__min_samples_split": Integer(2, 50),
            "model__min_samples_leaf": Integer(1, 20),
            "model__criterion": Categorical(["gini", "entropy"]),
        },
    },
    "random_forest": {
        "estimator": RandomForestClassifier(class_weight="balanced", n_jobs=1, random_state=RANDOM_STATE),
        "search_space": {
            "model__n_estimators": Integer(200, 1200),
            "model__max_depth": Integer(2, 40),
            "model__min_samples_split": Integer(2, 50),
            "model__min_samples_leaf": Integer(1, 20),
            "model__max_features": Real(0.2, 1.0),
            "model__bootstrap": Categorical([True, False]),
        },
    },
    "adaboost": {
        "estimator": AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
            random_state=RANDOM_STATE,
        ),
        "search_space": {
            "model__n_estimators": Integer(50, 600),
            "model__learning_rate": Real(1e-3, 2.0, prior="log-uniform"),
        },
    },
    "xgboost": {
        "estimator": XGBClassifier(
            objective="binary:logistic", eval_metric="auc",
            tree_method="hist", n_jobs=1, random_state=RANDOM_STATE,
            scale_pos_weight=CLASS_RATIO_TRAIN,
        ),
        "search_space": {
            "model__n_estimators": Integer(200, 1500),
            "model__max_depth": Integer(2, 10),
            "model__learning_rate": Real(1e-3, 0.3, prior="log-uniform"),
            "model__subsample": Real(0.5, 1.0),
            "model__colsample_bytree": Real(0.5, 1.0),
            "model__min_child_weight": Real(0.5, 20.0, prior="log-uniform"),
            "model__gamma": Real(0.0, 5.0),
            "model__reg_lambda": Real(1e-3, 100.0, prior="log-uniform"),
        },
    },
    "catboost": {
        "estimator": SafeCatBoostClassifier(
            loss_function="Logloss", eval_metric="AUC",
            auto_class_weights="Balanced",
            random_state=RANDOM_STATE, verbose=False,
            allow_writing_files=False, thread_count=1,
        ),
        "search_space": {
            "model__iterations": Integer(200, 2000),
            "model__depth": Integer(4, 10),
            "model__learning_rate": Real(1e-3, 0.3, prior="log-uniform"),
            "model__l2_leaf_reg": Real(1e-3, 100.0, prior="log-uniform"),
        },
    },
}

MODELS_WITH_SAMPLE_WEIGHT = {"adaboost"}
print("Modelos:", list(model_specs.keys()))


In [ ]:
outer_cv = RepeatedStratifiedKFold(
    n_splits=OUTER_SPLITS, n_repeats=OUTER_REPEATS, random_state=RANDOM_STATE
)
inner_cv = StratifiedKFold(
    n_splits=INNER_SPLITS, shuffle=True, random_state=RANDOM_STATE
)

selection_rows = []
selection_details = {}

for model_name, spec in model_specs.items():
    print("\nModelo:", model_name)

    fold_metrics_05 = []
    fold_best_params = []
    oof_proba = np.full(len(y_train), -1.0, dtype=float)

    for fold_i, (tr_idx, te_idx) in enumerate(outer_cv.split(X_train, y_train), start=1):
        X_tr = X_train.iloc[tr_idx]
        X_te = X_train.iloc[te_idx]
        y_tr = y_train[tr_idx]
        y_te = y_train[te_idx]

        pipe = make_pipeline(spec["estimator"])

        fit_params = {}
        if model_name in MODELS_WITH_SAMPLE_WEIGHT:
            fit_params["model__sample_weight"] = sw_train[tr_idx]

        search = BayesSearchCV(
            estimator=pipe,
            search_spaces=spec["search_space"],
            n_iter=N_ITER_BAYES,
            scoring=OPTIMIZE_METRIC,
            cv=inner_cv,
            n_jobs=N_JOBS_BAYES,
            refit=True,
            random_state=RANDOM_STATE + fold_i,
            verbose=VERBOSE_BAYES,
        )
        search.fit(X_tr, y_tr, **fit_params)

        proba = search.best_estimator_.predict_proba(X_te)[:, 1]
        pred_05 = (proba >= 0.5).astype(int)
        m05 = compute_metrics(y_te, proba, pred_05)

        fold_metrics_05.append(m05)
        fold_best_params.append(search.best_params_)
        oof_proba[te_idx] = proba

        print(f"  fold {fold_i:02d} | pr_auc={m05['average_precision']:.4f} | roc_auc={m05['roc_auc']:.4f}")

    df_folds = pd.DataFrame(fold_metrics_05)
    mean_05 = df_folds.mean(numeric_only=True)
    std_05 = df_folds.std(numeric_only=True)

    t_star, _ = select_threshold(y_train, oof_proba, THRESH_METRIC, MIN_RECALL)
    pred_oof = (oof_proba >= t_star).astype(int)
    oof_m = compute_metrics(y_train, oof_proba, pred_oof)

    selection_details[model_name] = {
        "metrics_mean_t05": mean_05.to_dict(),
        "metrics_std_t05": std_05.to_dict(),
        "best_params_per_fold": fold_best_params,
        "oof_threshold": t_star,
        "oof_metrics": oof_m,
    }

    selection_rows.append({
        "modelo": model_name,
        "pr_auc_mean": float(mean_05["average_precision"]),
        "pr_auc_std": float(std_05["average_precision"]),
        "roc_auc_mean": float(mean_05["roc_auc"]),
        "roc_auc_std": float(std_05["roc_auc"]),
        "brier_mean": float(mean_05["brier"]),
        "f1@0.5_mean": float(mean_05["f1"]),
        "recall@0.5_mean": float(mean_05["recall"]),
        "precision@0.5_mean": float(mean_05["precision"]),
        "bal_acc@0.5_mean": float(mean_05["balanced_accuracy"]),
        "mcc@0.5_mean": float(mean_05["mcc"]),
        "oof_threshold": t_star,
        "oof_f1": float(oof_m.get("f1", 0)),
        "oof_recall": float(oof_m.get("recall", 0)),
        "oof_precision": float(oof_m.get("precision", 0)),
        "oof_mcc": float(oof_m.get("mcc", 0)),
    })


In [ ]:
from IPython.display import display

selection_table = (
    pd.DataFrame(selection_rows)
    .sort_values(["pr_auc_mean", "roc_auc_mean", "brier_mean"], ascending=[False, False, True])
    .reset_index(drop=True)
)
selection_table.index += 1
selection_table.index.name = "rank"

display(selection_table)

winner_model = selection_table.iloc[0]["modelo"]
print("\nVencedor:", winner_model)


In [ ]:
spec = model_specs[winner_model]
rs_eval = RANDOM_STATE + 777

pipe_eval = make_pipeline(spec["estimator"])

fit_params_eval = {}
if winner_model in MODELS_WITH_SAMPLE_WEIGHT:
    fit_params_eval["model__sample_weight"] = sw_train

search_eval = BayesSearchCV(
    estimator=pipe_eval,
    search_spaces=spec["search_space"],
    n_iter=N_ITER_BAYES,
    scoring=OPTIMIZE_METRIC,
    cv=StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=rs_eval),
    n_jobs=N_JOBS_BAYES,
    refit=True,
    random_state=rs_eval,
    verbose=VERBOSE_BAYES,
)
search_eval.fit(X_train, y_train, **fit_params_eval)

print("Best score:", round(search_eval.best_score_, 4))
print("Best params:", search_eval.best_params_)

proba_oof_eval = oof_predictions(search_eval.best_estimator_, X_train, y_train, INNER_SPLITS, rs_eval)
t_star_eval, t_val_eval = select_threshold(y_train, proba_oof_eval, THRESH_METRIC, MIN_RECALL)
print(f"Threshold OOF: {t_star_eval:.3f} ({THRESH_METRIC}={t_val_eval:.4f})")

proba_holdout = search_eval.best_estimator_.predict_proba(X_test)[:, 1]
pred_holdout = (proba_holdout >= t_star_eval).astype(int)
holdout_metrics = compute_metrics(y_test, proba_holdout, pred_holdout)

pred_oof_eval = (proba_oof_eval >= t_star_eval).astype(int)
oof_m_eval = compute_metrics(y_train, proba_oof_eval, pred_oof_eval)

diag = pd.DataFrame({
    "Métrica": list(holdout_metrics.keys()),
    "Treino (OOF)": [f"{oof_m_eval[k]:.4f}" for k in holdout_metrics],
    "Teste (Holdout)": [f"{holdout_metrics[k]:.4f}" for k in holdout_metrics],
    "Gap": [f"{(oof_m_eval[k] - holdout_metrics[k]):+.4f}" for k in holdout_metrics],
})
display(diag)


In [ ]:
ci_dict = bootstrap_ci(
    y_test, proba_holdout, pred_holdout,
    n_boot=N_BOOTSTRAPS, ci=CI_LEVEL, random_state=RANDOM_STATE
)

ci_df = pd.DataFrame(ci_dict).T
ci_df.columns = ["Média (boot)", f"IC {CI_LEVEL*100:.0f}% inf", f"IC {CI_LEVEL*100:.0f}% sup"]
display(ci_df.round(4))


In [ ]:
if SAVE_CSV:
    selection_table.to_csv(os.path.join(OUTPUT_DIR, "ranking_modelos.csv"), encoding="utf-8")
    print("CSV ranking salvo.")

if SAVE_JSON:
    payload = {
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "config": {
            "base_file": BASE_FILE,
            "target_col": TARGET_COL,
            "test_size": TEST_SIZE,
            "random_state": RANDOM_STATE,
            "class_ratio_train": CLASS_RATIO_TRAIN,
            "outer_splits": OUTER_SPLITS,
            "outer_repeats": OUTER_REPEATS,
            "inner_splits": INNER_SPLITS,
            "n_iter_bayes": N_ITER_BAYES,
            "optimize_metric": OPTIMIZE_METRIC,
            "thresh_metric": THRESH_METRIC,
            "min_recall": MIN_RECALL,
            "models": list(model_specs.keys()),
            "pipeline": "Selecao Modelo",
        },
        "winner_model": winner_model,
        "ranking": selection_table.reset_index().to_dict(orient="records"),
        "model_details": {
            name: {
                "metrics_mean_t05": d["metrics_mean_t05"],
                "metrics_std_t05": d["metrics_std_t05"],
                "oof_threshold": d["oof_threshold"],
                "oof_metrics": d["oof_metrics"],
                "best_params_per_fold": d["best_params_per_fold"],
            }
            for name, d in selection_details.items()
        },
        "holdout_evaluation": {
            "model": winner_model,
            "threshold": t_star_eval,
            "metrics": holdout_metrics,
            "oof_metrics_eval": oof_m_eval,
            "bootstrap_ci": ci_dict,
            "best_params": search_eval.best_params_,
        },
        "holdout_split": split_info,
    }

    json_path = os.path.join(OUTPUT_DIR, "resultados_selecao_modelo.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, default=str)
    print("JSON salvo:", json_path)

print("Concluído. Pasta:", OUTPUT_DIR)
